Step 1: Doing Setup

In [ ]:
!pip install openai
!pip install langchain
!pip install faiss-cpu
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.2 MB/s eta 0:00:00


Step 2: Create Knowledge Base (RAG)

In [ ]:
product_info = {
  "pricing": {
    "basic": {
      "price": "$29/month",
      "videos": "10 videos/month",
      "quality": "720p"
    },
    "pro": {
      "price": "$79/month",
      "videos": "Unlimited",
      "quality": "4K",
      "features": ["AI captions", "24/7 support"]
    }
  },
  "policies": [
    "No refunds after 7 days",
    "24/7 support only for Pro plan"
  ]
}

Step 3: Intent Detection

In [ ]:
def detect_intent(user_input):
    """Understand what the user is really trying to say"""
    message = user_input.lower()

    # User just saying hello
    greetings = ["what's up","hi", "hello", "hey", "sup", "howdy"]
    if any(word in message for word in greetings):
        return "greeting"

    # Asking about money stuff
    pricing_keywords = ["price", "cost", "plan", "pricing", "how much", "subscription fee"]
    if any(word in message for word in pricing_keywords):
        return "pricing"

    # Ready to take action
    buying_keywords = ["buy", "subscribe", "start", "try", "purchase", "sign up", "get started"]
    if any(word in message for word in buying_keywords):
        return "high_intent"

    # Everything else
    return "other"

Step 4: RAG Response

In [ ]:
def answer_their_question(query):
    global product_info
    if "price" in query.lower() or "pricing" in query.lower() or "cost" in query.lower():
        basic_plan = product_info["pricing"]["basic"]
        pro_plan = product_info["pricing"]["pro"]
        return f"Our Basic plan costs {basic_plan['price']} for {basic_plan['videos']} at {basic_plan['quality']} quality. The Pro plan is {pro_plan['price']} for {pro_plan['videos']} at {pro_plan['quality']} quality, and includes {', '.join(pro_plan['features'])}. Do you want to know about our policies?"

    if "policies" in query.lower():
        return "Here are our policies: " + ", ".join(product_info["policies"])

    return "I can tell you about our pricing and policies. What would you like to know?"

Step 5: Tool Execution

In [ ]:
def mock_lead_capture(name, email, platform):
    print(f"Lead captured successfully: {name}, {email}, {platform}")

Step 6: Conversation Flow

In [ ]:
conversation_memory = {}

def start_chat():
    print("Bot: Hey there! Type 'quit' anytime to exit")

    while True:
        # Get what the user says
        user_says = input("\nYou: ")

        # Let them leave if they want
        if user_says.lower() in ['quit', 'exit', 'bye']:
            print("Bot: Talk to you later! 👋")
            break

        # Figure out what they're really asking
        what_they_want = detect_intent(user_says)

        # Say hello back
        if what_they_want == "greeting":
            print("Bot: Hey! What brings you here today?")

        # Talk about money stuff
        elif what_they_want == "pricing":
            print("Bot:", answer_their_question(user_says))

        # They sound ready to buy
        elif what_they_want == "high_intent":
            print("Bot: Awesome! Let's get you set up 🎉")

            # Grab their details
            conversation_memory["full_name"] = input("Bot: Quick one - what should I call you? ")
            conversation_memory["email_address"] = input("Bot: And your email? ")
            conversation_memory["content_platform"] = input("Bot: Cool! Where do you usually post your content? ")

            # Save their info
            mock_lead_capture(
                conversation_memory["full_name"],
                conversation_memory["email_address"],
                conversation_memory["content_platform"]
            )

            print("Bot: Perfect! Someone from our team will email you soon. Anything else I can help with?")

        # When we're confused
        else:
            print("Bot: Hmm, I'm not sure I got that. Could you rephrase?")

In [ ]:
start_chat()

Bot: Hey there! 👋 Type 'quit' anytime to exit

You: hi
Bot: Hey! What brings you here today?

You: tell me about your pricing
Bot: Our Basic plan costs $29/month for 10 videos/month at 720p quality. The Pro plan is $79/month for Unlimited at 4K quality, and includes AI captions, 24/7 support. Do you want to know about our policies?

You: That sounds good, I want to try the Pro plan pricing for my YouTube channel
Bot: Our Basic plan costs $29/month for 10 videos/month at 720p quality. The Pro plan is $79/month for Unlimited at 4K quality, and includes AI captions, 24/7 support. Do you want to know about our policies?

You: subscribe
Bot: Awesome! Let's get you set up 🎉
Bot: Quick one - what should I call you? Harsh Vardhan Singh
Bot: And your email? harshsingh8400@gmail.com
Bot: Cool! Where do you usually post your content? Github
Lead captured successfully: Harsh Vardhan Singh, harshsingh8400@gmail.com, Github
Bot: Perfect! Someone from our team will email you soon. Anything else I can

Memory

In [1]:
conversation_history = []

def chat():
    while True:
        user_input = input("User: ")
        conversation_history.append(user_input)

        # keep only last 6 messages
        if len(conversation_history) > 6:
            conversation_history.pop(0)